In [27]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import models, layers, preprocessing
from sklearn.metrics import classification_report, confusion_matrix


In [45]:

data_dir = "Downloads/archive/kagglecatsanddogs_3367a/PetImages"
img_size = (64, 64)

datagen = preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=32,
    class_mode='categorical',S
    subset='training'
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)


Found 19968 images belonging to 2 classes.
Found 4991 images belonging to 2 classes.


In [47]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    layers.MaxPooling2D(2,2),
    
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),   # Regularization
    layers.Dense(2, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [52]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=1
)


624/624 ━━━━━━━━━━━━━━━━━━━━ 78s 124ms/step - accuracy: 0.9522 - loss: 0.1192 - val_accuracy: 0.8581 - val_loss: 0.4347


In [54]:
# Accuracy
loss, acc = model.evaluate(val_gen)
print(f"Validation Accuracy: {acc:.4f}")

# Predictions
y_pred = model.predict(val_gen)
y_pred_classes = np.argmax(y_pred, axis=1)

# True labels
y_true = val_gen.classes

# Classification Report
print(classification_report(y_true, y_pred_classes, target_names=["Cat","Dog"]))


156/156 ━━━━━━━━━━━━━━━━━━━━ 12s 74ms/step - accuracy: 0.8581 - loss: 0.4347
Validation Accuracy: 0.8581
156/156 ━━━━━━━━━━━━━━━━━━━━ 12s 77ms/step
              precision    recall  f1-score   support

         Cat       0.88      0.83      0.85      2498
         Dog       0.84      0.88      0.86      2493

    accuracy                           0.86      4991
   macro avg       0.86      0.86      0.86      4991
weighted avg       0.86      0.86      0.86      4991



In [56]:
cm = confusion_matrix(y_true, y_pred_classes)
print("Confusion Matrix:\n", cm)


Confusion Matrix:
 [[2079  419]
 [ 289 2204]]


In [62]:
model.save("cat_dog_model.keras")